# Multimodal Fusion for Herpes Zoster Detection

**Part of:** *A Multimodal Explainable AI Framework for Herpes Zoster Detection* (7th-Semester B.Tech Project)

## 1. Objective

This notebook implements the **fusion stage** of the multimodal pipeline. It does **not** retrain
either upstream branch. It assumes:

- The **image branch** (ResNet50) has already been trained, and case-level image embeddings
  (2048-D, averaged across a case's images) have been extracted and saved.
- The **clinical/metadata branch** (clinical encoder) has already been trained, and case-level
  clinical embeddings (16-D) have been extracted and saved.
- Both branches used the **same fixed, case-level train/validation/test split**
  (78 / 17 / 17 cases; 112 total cases, 56 HZ / 56 Non-HZ, 223 total images).

**Goal of this notebook:** test whether concatenating the already-learned image and clinical
representations (feature-level intermediate fusion, 2048 + 16 = 2064-D) provides useful
complementary information beyond either modality alone, using:

1. A **Logistic Regression** fusion baseline.
2. A **small, regularized neural fusion classifier**.

This is a research / decision-support prototype exploring representation complementarity.
It is **not** a diagnostic tool, and no clinical claims are made from its results. Given the very
small test set (17 cases), all reported metrics carry substantial statistical uncertainty and
should be interpreted as indicative, not conclusive.


## 0. Setup

Imports, global configuration, and reproducibility settings. Notebook is assumed to be executed
from the **repository root** (`herpes_zoster_btp/`). No absolute or machine-specific paths are used.


In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
)

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

import joblib

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


In [ ]:
# ------------------------------------------------------------------
# Paths (relative to repository root -- do not use absolute paths)
# ------------------------------------------------------------------
REPO_ROOT = Path(".")

IMAGE_EMB_DIR = REPO_ROOT / "outputs" / "image" / "embeddings"
CLINICAL_EMB_DIR = REPO_ROOT / "outputs" / "clinical" / "embeddings"

FUSION_OUT_DIR = REPO_ROOT / "outputs" / "fusion"
FUSION_PLOTS_DIR = FUSION_OUT_DIR / "plots"
FUSION_MODELS_DIR = FUSION_OUT_DIR / "models"

for d in [FUSION_OUT_DIR, FUSION_PLOTS_DIR, FUSION_MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

IMAGE_FILES = {
    "train": IMAGE_EMB_DIR / "train_case_embeddings.csv",
    "validation": IMAGE_EMB_DIR / "validation_case_embeddings.csv",
    "test": IMAGE_EMB_DIR / "test_case_embeddings.csv",
}

CLINICAL_FILES = {
    "train": CLINICAL_EMB_DIR / "clinical_train.csv",
    "validation": CLINICAL_EMB_DIR / "clinical_validation.csv",
    "test": CLINICAL_EMB_DIR / "clinical_test.csv",
}

IMAGE_EMB_COLS = [f"emb_{i}" for i in range(2048)]
CLINICAL_EMB_COLS = [f"clinical_embed_{i}" for i in range(16)]
FUSED_FEATURE_COLS = IMAGE_EMB_COLS + CLINICAL_EMB_COLS

EXPECTED_CASE_COUNTS = {"train": 78, "validation": 17, "test": 17}

assert len(IMAGE_EMB_COLS) == 2048
assert len(CLINICAL_EMB_COLS) == 16
assert len(FUSED_FEATURE_COLS) == 2064
print("Expected fused feature dimension:", len(FUSED_FEATURE_COLS))


## 2. Loading Verified Embeddings

Load the already-generated, already-verified case-level embeddings from both branches. These are
loaded **as-is**: no re-aggregation, no re-training, and no changes to the fixed split. Image
embeddings were already averaged per case by the image branch; `pred_prob` from the image branch
is loaded for reference only and is **not** used as a fusion feature.


In [ ]:
image_dfs = {split: pd.read_csv(path) for split, path in IMAGE_FILES.items()}
clinical_dfs = {split: pd.read_csv(path) for split, path in CLINICAL_FILES.items()}

for split in ["train", "validation", "test"]:
    print(f"[image]    {split:<10s} shape = {image_dfs[split].shape}")
    print(f"[clinical] {split:<10s} shape = {clinical_dfs[split].shape}")


In [ ]:
# Sanity check: required columns are present in each file, and pred_prob is confirmed
# present but will be explicitly excluded from the fusion feature set below.
for split in ["train", "validation", "test"]:
    img_cols = set(image_dfs[split].columns)
    clin_cols = set(clinical_dfs[split].columns)

    assert {"case_id", "true_label", "pred_prob"}.issubset(img_cols), f"{split}: missing expected image columns"
    assert {"case_id", "label"}.issubset(clin_cols), f"{split}: missing expected clinical columns"

    missing_img_emb = [c for c in IMAGE_EMB_COLS if c not in img_cols]
    missing_clin_emb = [c for c in CLINICAL_EMB_COLS if c not in clin_cols]
    assert not missing_img_emb, f"{split}: missing image embedding columns: {missing_img_emb[:5]}..."
    assert not missing_clin_emb, f"{split}: missing clinical embedding columns: {missing_clin_emb}"

print("All expected columns present in all splits. 'pred_prob' will NOT be used as a fusion feature.")


## 3. Alignment by `case_id`

**Critical:** the row order of the image and clinical embedding CSVs is not guaranteed to match.
We therefore perform an explicit **one-to-one merge on `case_id`** for every split, rather than
concatenating by row position. After merging we assert that the clinical `label` agrees with the
image `true_label` for every case, confirming the two modalities refer to the same case and the
same ground truth.


In [ ]:
merged_dfs = {}

for split in ["train", "validation", "test"]:
    img_df = image_dfs[split][["case_id", "true_label"] + IMAGE_EMB_COLS].copy()
    clin_df = clinical_dfs[split][["case_id", "label"] + CLINICAL_EMB_COLS].copy()

    # Explicit, position-independent, one-to-one merge on case_id.
    merged = clin_df.merge(img_df, on="case_id", how="inner", validate="one_to_one")

    # No cases should be dropped by the merge -- every case_id must exist in both modalities.
    assert len(merged) == len(clin_df) == len(img_df), (
        f"{split}: merge dropped rows (clinical={len(clin_df)}, image={len(img_df)}, "
        f"merged={len(merged)}). Some case_ids do not exist in both modalities."
    )

    # Ground-truth label must agree across modalities for every case.
    assert (merged["label"] == merged["true_label"]).all(), (
        f"{split}: label mismatch between clinical and image branch for one or more cases"
    )

    merged["label"] = merged["label"].astype(int)
    merged = merged.drop(columns=["true_label"])

    merged_dfs[split] = merged
    print(f"{split}: merged shape = {merged.shape} (expected {EXPECTED_CASE_COUNTS[split]} cases)")


## 4. Leakage and Integrity Checks

Before building any fusion features, we run a full set of integrity assertions:

- `case_id` is unique within each modality and split.
- No matched cases are missing after merging (checked above).
- Train / validation / test case IDs **do not overlap** (no case leaks across splits).
- No `NaN` or infinite values exist anywhere in the fused embeddings.
- Exactly 16 clinical embedding features and 2048 image embedding features exist.
- The concatenated fusion feature dimension is exactly 2064.
- Merged case counts match the fixed split (78 / 17 / 17).

If any assertion fails, the notebook stops here rather than silently proceeding with corrupted data.


In [ ]:
# --- Per-modality uniqueness (checked pre-merge, re-verified here) ---
for split in ["train", "validation", "test"]:
    assert image_dfs[split]["case_id"].is_unique, f"{split}: duplicate case_id in image embeddings"
    assert clinical_dfs[split]["case_id"].is_unique, f"{split}: duplicate case_id in clinical embeddings"

# --- No missing matched cases across modalities ---
for split in ["train", "validation", "test"]:
    img_ids = set(image_dfs[split]["case_id"])
    clin_ids = set(clinical_dfs[split]["case_id"])
    unmatched = img_ids.symmetric_difference(clin_ids)
    assert not unmatched, f"{split}: case_ids present in only one modality: {unmatched}"

# --- Train / validation / test case_id do not overlap ---
train_ids = set(merged_dfs["train"]["case_id"])
val_ids = set(merged_dfs["validation"]["case_id"])
test_ids = set(merged_dfs["test"]["case_id"])

assert not (train_ids & val_ids), "Data leakage: train/validation case_id overlap detected"
assert not (train_ids & test_ids), "Data leakage: train/test case_id overlap detected"
assert not (val_ids & test_ids), "Data leakage: validation/test case_id overlap detected"

# --- Feature dimensionality ---
assert len(IMAGE_EMB_COLS) == 2048, "Expected exactly 2048 image embedding features"
assert len(CLINICAL_EMB_COLS) == 16, "Expected exactly 16 clinical embedding features"
assert len(FUSED_FEATURE_COLS) == 2064, "Expected exactly 2064 fused features"

# --- No NaN / infinite values, and expected case counts ---
for split in ["train", "validation", "test"]:
    feat_matrix = merged_dfs[split][FUSED_FEATURE_COLS].to_numpy(dtype=np.float64)
    assert np.isfinite(feat_matrix).all(), f"{split}: NaN or infinite values found in fused embeddings"
    assert len(merged_dfs[split]) == EXPECTED_CASE_COUNTS[split], (
        f"{split}: expected {EXPECTED_CASE_COUNTS[split]} cases, found {len(merged_dfs[split])}"
    )

print("All integrity and leakage checks passed:")
print(f"  - Train/Val/Test case_id overlap: none")
print(f"  - Fused feature dimension: {len(FUSED_FEATURE_COLS)} (2048 image + 16 clinical)")
print(f"  - Case counts -> train: {len(merged_dfs['train'])}, "
      f"validation: {len(merged_dfs['validation'])}, test: {len(merged_dfs['test'])}")
for split in ["train", "validation", "test"]:
    print(f"  - {split} label balance:", merged_dfs[split]["label"].value_counts().to_dict())


## 5. Feature-Level Fusion

We use **feature-level (early/intermediate) fusion**: the 2048-D image embedding and 16-D
clinical embedding for each case are concatenated into a single 2064-D vector. This keeps the
fusion step simple and lets both downstream classifiers (Logistic Regression and the neural
fusion model) learn directly from the combined representation, which is appropriate given the
already-small number of cases -- there is not enough data here to justify a more elaborate,
learned fusion mechanism (e.g. attention-based fusion) without a high risk of overfitting.


In [ ]:
def build_xy(df: pd.DataFrame):
    X = df[FUSED_FEATURE_COLS].to_numpy(dtype=np.float32)
    y = df["label"].to_numpy(dtype=np.int64)
    case_ids = df["case_id"].to_numpy()
    return X, y, case_ids

X_train, y_train, ids_train = build_xy(merged_dfs["train"])
X_val, y_val, ids_val = build_xy(merged_dfs["validation"])
X_test, y_test, ids_test = build_xy(merged_dfs["test"])

print("X_train:", X_train.shape, " y_train:", y_train.shape)
print("X_val:  ", X_val.shape, " y_val:  ", y_val.shape)
print("X_test: ", X_test.shape, " y_test: ", y_test.shape)


In [ ]:
# Persist the merged, fused datasets for reproducibility. These preserve the fixed split
# and retain case_id + label alongside the fused features.
for split, df in merged_dfs.items():
    out_cols = ["case_id", "label"] + FUSED_FEATURE_COLS
    df[out_cols].to_csv(FUSION_OUT_DIR / f"fused_{split}.csv", index=False)

print("Saved merged fusion datasets to:", FUSION_OUT_DIR)


## 6. Logistic Regression Fusion Baseline

A simple, strongly-regularized Logistic Regression on the 2064-D fused features serves as the
fusion baseline. The scaler is **fit only on the training set** and used to transform validation
and test data; no leakage of validation/test statistics into preprocessing or model fitting.
Regularization strength (`C`) is fixed a priori (not tuned against validation or test performance)
to keep the baseline simple, given the very small training set (78 cases for 2064 features).


In [ ]:
scaler = StandardScaler()
scaler.fit(X_train)  # fit ONLY on training data

X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, FUSION_MODELS_DIR / "fusion_scaler.joblib")
print("Scaler fit on training data only; saved to", FUSION_MODELS_DIR / "fusion_scaler.joblib")


In [ ]:
# Fixed, reasonably strong L2 regularization -- not tuned on validation/test.
# With only 78 training cases and 2064 features, strong regularization is essential.
LOGREG_C = 0.05

logreg = LogisticRegression(
    C=LOGREG_C,
    solver="lbfgs",
    max_iter=5000,
    random_state=SEED,
)
logreg.fit(X_train_scaled, y_train)

joblib.dump(logreg, FUSION_MODELS_DIR / "logreg_fusion_model.joblib")
print(f"Logistic Regression fusion model fit (C={LOGREG_C}); saved to {FUSION_MODELS_DIR}")


In [ ]:
def evaluate_predictions(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else float("nan")
    metrics = {
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall_sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob) if len(set(y_true)) > 1 else float("nan"),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
    }
    return metrics, y_pred

# Validation performance is used only for monitoring/model selection, per the fixed 0.5 threshold.
logreg_val_prob = logreg.predict_proba(X_val_scaled)[:, 1]
logreg_val_metrics, _ = evaluate_predictions(y_val, logreg_val_prob, threshold=0.5)

print("Logistic Regression -- VALIDATION metrics (threshold = 0.5):")
for k, v in logreg_val_metrics.items():
    print(f"  {k}: {v}")


## 7. Neural Fusion Model

A small, heavily-regularized feed-forward network on the 2064-D fused vector:

```
2064 -> Linear(2064, 128) -> ReLU -> Dropout -> Linear(128, 32) -> ReLU -> Dropout -> Linear(32, 1)
```

Given only 78 training cases, this architecture is intentionally kept compact. Dropout and weight
decay provide the main regularization. `BCEWithLogitsLoss` is used with a single logit output.


In [ ]:
class FusionMLP(nn.Module):
    def __init__(self, in_dim=2064, hidden1=128, hidden2=32, dropout=0.5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden2, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

n_params = sum(p.numel() for p in FusionMLP().parameters())
print(f"FusionMLP parameter count: {n_params:,} (kept small relative to 78 training cases)")


In [ ]:
def to_tensor_dataset(X, y):
    return TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32),
    )

BATCH_SIZE = 16

# NOTE: only train/validation datasets and loaders are created at this stage.
# The test set is intentionally not wrapped into a Dataset/DataLoader here to keep
# it fully separate from anything touched during training and model selection.
# test_ds / test_loader are created later, in Section 9, only after training,
# early stopping, and best-checkpoint restoration are complete.
train_ds = to_tensor_dataset(X_train_scaled, y_train)
val_ds = to_tensor_dataset(X_val_scaled, y_val)

g = torch.Generator()
g.manual_seed(SEED)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, generator=g)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)


## 8. Training with Validation-Based Model Selection

The neural fusion model is trained with:

- `BCEWithLogitsLoss`
- Adam optimizer with weight decay (L2 regularization)
- **Early stopping** and **checkpoint selection based on validation loss only**
- The **best validation-loss checkpoint** is restored before any test-set evaluation

Test data is not touched anywhere in this section.


In [ ]:
MAX_EPOCHS = 200
PATIENCE = 20
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-3
DROPOUT = 0.5

torch.manual_seed(SEED)
model = FusionMLP(in_dim=X_train_scaled.shape[1], dropout=DROPOUT).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

best_val_loss = float("inf")
best_state = None
best_epoch = -1
epochs_without_improvement = 0

history = {"train_loss": [], "val_loss": []}

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    running_loss = 0.0
    n_train = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xb.size(0)
        n_train += xb.size(0)
    train_loss = running_loss / n_train

    model.eval()
    running_val_loss = 0.0
    n_val = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            loss = criterion(logits, yb)
            running_val_loss += loss.item() * xb.size(0)
            n_val += xb.size(0)
    val_loss = running_val_loss / n_val

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    improved = val_loss < best_val_loss - 1e-6
    if improved:
        best_val_loss = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        best_epoch = epoch
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epoch == 1 or epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}"
              f"{'  <-- best' if improved else ''}")

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping at epoch {epoch} (no val_loss improvement for {PATIENCE} epochs).")
        break

print(f"\nBest validation loss = {best_val_loss:.4f} at epoch {best_epoch}")

# Restore the best validation checkpoint before any further evaluation.
model.load_state_dict(best_state)
model.eval()

torch.save(
    {"model_state_dict": best_state, "best_epoch": best_epoch, "best_val_loss": best_val_loss,
     "architecture": {"in_dim": X_train_scaled.shape[1], "hidden1": 128, "hidden2": 32, "dropout": DROPOUT}},
    FUSION_MODELS_DIR / "neural_fusion_best.pt",
)
with open(FUSION_OUT_DIR / "training_history.json", "w") as f:
    json.dump(history, f, indent=2)

print("Best checkpoint restored and saved to", FUSION_MODELS_DIR / "neural_fusion_best.pt")


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(history["train_loss"], label="Train loss")
ax.plot(history["val_loss"], label="Validation loss")
ax.axvline(best_epoch - 1, color="gray", linestyle="--", linewidth=1, label=f"Best epoch ({best_epoch})")
ax.set_xlabel("Epoch")
ax.set_ylabel("BCE Loss")
ax.set_title("Neural Fusion Model -- Training History")
ax.legend()
fig.tight_layout()
fig.savefig(FUSION_PLOTS_DIR / "training_history.png", dpi=150)
plt.show()


In [ ]:
@torch.no_grad()
def get_probs(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    for xb, yb in loader:
        xb = xb.to(DEVICE)
        logits = model(xb)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(yb.numpy())
    return np.concatenate(all_probs), np.concatenate(all_labels)

nn_val_prob, nn_val_y = get_probs(model, val_loader)
nn_val_metrics, _ = evaluate_predictions(nn_val_y, nn_val_prob, threshold=0.5)

print("Neural Fusion Model -- VALIDATION metrics (threshold = 0.5, best checkpoint):")
for k, v in nn_val_metrics.items():
    print(f"  {k}: {v}")


## 9. Final Test Evaluation

**Test data is evaluated only now, after both models and the neural network's checkpoint have
already been fully selected using training/validation data alone.** No architecture, hyperparameter,
epoch, or threshold choices were informed by test performance. A fixed threshold of 0.5 is used for
both models, matching the validation-time threshold, for simplicity and comparability.


In [ ]:
# --- Logistic Regression: final test evaluation ---
logreg_test_prob = logreg.predict_proba(X_test_scaled)[:, 1]
logreg_test_metrics, logreg_test_pred = evaluate_predictions(y_test, logreg_test_prob, threshold=0.5)

print("Logistic Regression -- TEST metrics (threshold = 0.5):")
for k, v in logreg_test_metrics.items():
    print(f"  {k}: {v}")


In [ ]:
# Test dataset/loader are created only now -- after neural-network training,
# early stopping, and best-validation-checkpoint restoration are fully complete.
# No architecture, hyperparameter, epoch, or checkpoint decision has used test data.
test_ds = to_tensor_dataset(X_test_scaled, y_test)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)


In [ ]:
# --- Neural Fusion Model: final test evaluation (best validation checkpoint) ---
nn_test_prob, nn_test_y = get_probs(model, test_loader)
nn_test_metrics, nn_test_pred = evaluate_predictions(nn_test_y, nn_test_prob, threshold=0.5)

print("\nNeural Fusion Model -- TEST metrics (threshold = 0.5, best validation checkpoint):")
for k, v in nn_test_metrics.items():
    print(f"  {k}: {v}")


## 10. ROC Curves and Confusion Matrices (Test Set)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

for ax, (name, y_prob, y_pred) in zip(
    axes,
    [("Logistic Regression", logreg_test_prob, logreg_test_pred),
     ("Neural Fusion Model", nn_test_prob, nn_test_pred)],
):
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(f"{name}\nTest Confusion Matrix")
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Non-HZ", "HZ"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Non-HZ", "HZ"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                     color="white" if cm[i, j] > cm.max() / 2 else "black")

fig.tight_layout()
fig.savefig(FUSION_PLOTS_DIR / "test_confusion_matrices.png", dpi=150)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

for name, y_prob in [("Logistic Regression", logreg_test_prob),
                      ("Neural Fusion Model", nn_test_prob)]:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")

ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Chance")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Test Set ROC Curves (n=17 -- interpret with caution)")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(FUSION_PLOTS_DIR / "test_roc_curve.png", dpi=150)
plt.show()


## 11. Results Comparison

The table below compares validation and test performance for both fusion models. **The test set
contains only 17 cases** (roughly 6% steps in accuracy per case), so small numeric differences
between models are not a reliable basis for claiming one model is superior. Both are reported for
transparency and completeness.


In [ ]:
comparison_rows = []
for name, val_m, test_m in [
    ("Logistic Regression", logreg_val_metrics, logreg_test_metrics),
    ("Neural Fusion Model", nn_val_metrics, nn_test_metrics),
]:
    comparison_rows.append({
        "model": name,
        "val_accuracy": val_m["accuracy"], "val_auc": val_m["roc_auc"],
        "val_f1": val_m["f1"],
        "test_accuracy": test_m["accuracy"], "test_auc": test_m["roc_auc"],
        "test_precision": test_m["precision"], "test_recall_sensitivity": test_m["recall_sensitivity"],
        "test_specificity": test_m["specificity"], "test_f1": test_m["f1"],
        "test_TN": test_m["TN"], "test_FP": test_m["FP"],
        "test_FN": test_m["FN"], "test_TP": test_m["TP"],
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df


In [ ]:
print("Note on interpretation:")
print("- The test set has only 17 cases; each case is worth ~5.9% of test accuracy.")
print("- Differences of 1-2 cases between models can easily flip which model looks 'better'.")
print("- Neither model's test performance should be treated as a precise or stable estimate")
print("  of real-world performance; both should be viewed as exploratory evidence that fused")
print("  image + clinical representations are learnable from this dataset, not as proof that")
print("  one fusion approach is definitively superior to the other.")


### Saving Reproducibility Artifacts

All metrics, the fitted scaler, both trained models, training history, and plots are saved to
`outputs/fusion/` for reproducibility.


In [ ]:
metrics_payload = {
    "logistic_regression": {"validation": logreg_val_metrics, "test": logreg_test_metrics},
    "neural_fusion_model": {"validation": nn_val_metrics, "test": nn_test_metrics,
                             "best_epoch": best_epoch, "best_val_loss": best_val_loss},
    "seed": SEED,
    "fused_feature_dim": len(FUSED_FEATURE_COLS),
    "case_counts": EXPECTED_CASE_COUNTS,
}

with open(FUSION_OUT_DIR / "metrics.json", "w") as f:
    json.dump(metrics_payload, f, indent=2, default=float)

final_results = {
    "comparison_table": comparison_df.to_dict(orient="records"),
    "notes": [
        "Fixed case-level split reused from image and clinical branches (78/17/17).",
        "Threshold fixed at 0.5 for both models; not tuned on test data.",
        "Neural fusion model uses the best validation-loss checkpoint.",
        "Test set size (n=17) implies high variance in all reported test metrics.",
        "This is a research/decision-support prototype; not a diagnostic claim.",
    ],
}
with open(FUSION_OUT_DIR / "final_results.json", "w") as f:
    json.dump(final_results, f, indent=2, default=float)

print("Saved:")
print(" -", FUSION_OUT_DIR / "metrics.json")
print(" -", FUSION_OUT_DIR / "final_results.json")
print(" -", FUSION_MODELS_DIR / "fusion_scaler.joblib")
print(" -", FUSION_MODELS_DIR / "logreg_fusion_model.joblib")
print(" -", FUSION_MODELS_DIR / "neural_fusion_best.pt")
print(" -", FUSION_OUT_DIR / "training_history.json")
print(" -", FUSION_PLOTS_DIR / "training_history.png")
print(" -", FUSION_PLOTS_DIR / "test_confusion_matrices.png")
print(" -", FUSION_PLOTS_DIR / "test_roc_curve.png")
print(" -", FUSION_OUT_DIR / "fused_train.csv / fused_validation.csv / fused_test.csv")


## 12. Limitations and Interpretation

- **Sample size.** With only 78 training cases, 17 validation cases, and 17 test cases, all
  reported metrics (accuracy, AUC, sensitivity, specificity, etc.) have wide confidence intervals.
  A single case moving between correct and incorrect can shift test accuracy by roughly 6
  percentage points. These results should be read as **directional evidence**, not precise
  performance estimates.
- **Fusion strategy.** Only simple feature-level (concatenation) fusion was explored, evaluated
  with a linear baseline (Logistic Regression) and a small non-linear model (a 2-hidden-layer MLP).
  More sophisticated fusion mechanisms (e.g. attention-based or gated fusion) were intentionally
  avoided given the dataset size, since they would be prone to overfitting without a reliable way
  to validate them.
- **No hyperparameter search.** Regularization strength and network hyperparameters were fixed in
  advance rather than tuned, to avoid indirectly leaking validation/test information through
  repeated selection. This likely leaves some performance on the table but keeps the evaluation
  honest given the small validation set.
- **Threshold.** A fixed 0.5 classification threshold was used for both models rather than a
  validation-tuned threshold, for simplicity and comparability between models.
- **No clinical claims.** This notebook is a research / decision-support prototype exploring
  whether image and clinical representations are complementary. It is **not** a validated
  diagnostic tool and must not be used, or represented, as one. Any real-world deployment would
  require substantially larger and more diverse datasets, external validation, and clinical
  oversight.
- **Explainability.** This notebook focuses on fusion and evaluation only. Explainability analysis
  (e.g. Grad-CAM on the image branch, feature attribution on the clinical branch and/or fused
  representation) is expected to be handled in a separate, dedicated notebook, consistent with
  the "Explainable AI" framing of the overall project.
